# Stable-Baselines3 DQN으로 CartPole 테스트

위에서부터 순서대로 실행하세요. CartPole의 4차원 상태를 입력으로 사용하는 `MlpPolicy`를 학습합니다.
학습 후 별도 환경에서 10개 에피소드를 평가하고, 플레이를 애니메이션으로 확인합니다.

참고: [SB3 DQN 공식 문서](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html)


In [ ]:
%pip install stable-baselines3 "gymnasium[classic-control]" matplotlib


In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

ENV_ID = "CartPole-v1"
SEED = 42
TOTAL_TIMESTEPS = 100_000


## 학습

CPU에서 학습합니다. `TOTAL_TIMESTEPS`로 학습량을 조절할 수 있으며, 점수는 학습량과 시드에 따라 달라집니다.


In [2]:
train_env = Monitor(gym.make(ENV_ID))
try:
    model = DQN(
        "MlpPolicy",
        train_env,
        learning_rate=1e-3,
        buffer_size=50_000,
        learning_starts=1_000,
        batch_size=64,
        gamma=0.99,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=500,
        exploration_fraction=0.2,
        exploration_final_eps=0.05,
        policy_kwargs=dict(net_arch=[64, 64]),
        seed=SEED,
        device="mps",
        verbose=1,
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS, log_interval=20)
finally:
    train_env.close()


Using mps device
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 25.1     |
|    ep_rew_mean      | 25.1     |
|    exploration_rate | 0.976    |
| time/               |          |
|    episodes         | 20       |
|    fps              | 1718     |
|    time_elapsed     | 0        |
|    total_timesteps  | 501      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 23       |
|    ep_rew_mean      | 23       |
|    exploration_rate | 0.956    |
| time/               |          |
|    episodes         | 40       |
|    fps              | 3028     |
|    time_elapsed     | 0        |
|    total_timesteps  | 920      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21.2     |
|    ep_rew_mean      | 21.2     |
|    exploration_rate | 0.939    |
| t

## 평가

탐험 없이 학습한 정책으로 10회 평가합니다. CartPole-v1의 에피소드 최대 점수는 500입니다.


In [3]:
eval_env = Monitor(gym.make(ENV_ID))
try:
    eval_env.reset(seed=SEED + 1)
    mean_reward, std_reward = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=10,
        deterministic=True,
    )
    print(f"평균 보상: {mean_reward:.1f} ± {std_reward:.1f} / 500")
finally:
    eval_env.close()


평균 보상: 99.7 ± 3.7 / 500


## 플레이 확인

한 에피소드를 노트북 안에서 재생합니다.


In [4]:
render_env = gym.make(ENV_ID, render_mode="rgb_array")
frames = []
episode_reward = 0.0
try:
    obs, info = render_env.reset(seed=SEED + 2)
    frames.append(render_env.render())
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = render_env.step(int(action))
        episode_reward += reward
        frames.append(render_env.render())
        if terminated or truncated:
            break
finally:
    render_env.close()

print(f"플레이 보상: {episode_reward:.0f}")
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(frames[0])
ax.axis("off")

def update(frame):
    image.set_data(frame)
    return (image,)

# 매 두 프레임을 표시해 HTML 크기를 줄입니다 (원래 재생 속도 유지).
animation = FuncAnimation(fig, update, frames=frames[::2], interval=40)
plt.close(fig)
display(HTML(animation.to_jshtml()))


objc[19593]: Class SDL_RumbleMotor is implemented in both /Users/hoyeon/Codes/modularRNN/.venv/lib/python3.14/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11daccd40) and /Users/hoyeon/Codes/modularRNN/.venv/lib/python3.14/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12612c9c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[19593]: Class SDL_RumbleContext is implemented in both /Users/hoyeon/Codes/modularRNN/.venv/lib/python3.14/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11daccd90) and /Users/hoyeon/Codes/modularRNN/.venv/lib/python3.14/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12612ca18). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[19593]: Class SDLApplication is implemented in both /Users/hoyeon/Codes/modularRNN/.venv/lib/python3.14/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11dacc890) and /Users/hoyeon/C

플레이 보상: 99
